# News Topic Classifier — Fine-tuned BERT (AG News)

**Task 1 — AI/ML Engineering Advanced Internship, DevelopersHub Corporation**

**Objective:** Fine-tune `bert-base-uncased` on the AG News dataset to
classify news headlines into 4 topics (World, Sports, Business, Sci/Tech),
evaluate with accuracy/F1, and deploy for live interaction.

This notebook calls into the reusable modules under `src/` so the same
logic backs both this notebook and the Streamlit app.


## 1. Setup

In [ ]:
import sys
sys.path.append("..")  # so `import src...` works when running from notebooks/

import pandas as pd
import json
from src.config import LABEL_NAMES, RESULTS_DIR, MODEL_DIR


## 2. Dataset loading & preprocessing

AG News: 120,000 train / 7,600 test rows, 4 balanced classes.

In [ ]:
from src.data_utils import get_train_eval_splits

raw, tokenized = get_train_eval_splits()
print(f"Train rows: {len(raw['train'])}  |  Test rows: {len(raw['test'])}")
raw["train"][0]


In [ ]:
import pandas as pd
label_counts = pd.Series([LABEL_NAMES[i] for i in raw["train"]["label"]]).value_counts()
label_counts


## 3. Fine-tuning BERT

> This step downloads `bert-base-uncased` (~440MB) and fine-tunes it on AG
> News. On GPU, 3 epochs over the full dataset takes roughly 30-60 minutes;
> on CPU it will be much slower — use `--quick` mode via the command line
> (`python -m src.train --quick`) or subsample here for a faster notebook run.

We call the same `main()` logic as the command-line script by running it as
a module so training behavior stays identical whether you use the notebook
or the CLI.

In [ ]:
# Fast smoke-test settings (comment out to run the full training set)
import src.config as cfg
cfg.TRAIN_SUBSET_SIZE = 3000
cfg.EVAL_SUBSET_SIZE = 500


In [ ]:
from src.train import main as train_main
import sys as _sys
_sys.argv = ["train.py", "--epochs", "1"]  # remove this line to use full config defaults
train_main()


## 4. Evaluation on the test set

In [ ]:
from src.evaluate import run_evaluation

metrics, predictions_df = run_evaluation()
predictions_df.head(10)


## 5. Visualizations

In [ ]:
from src.visualize import generate_all_plots
generate_all_plots()


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(RESULTS_DIR / "figures" / "confusion_matrix.png")))


In [ ]:
display(Image(filename=str(RESULTS_DIR / "figures" / "per_class_metrics.png")))
display(Image(filename=str(RESULTS_DIR / "figures" / "confidence_distribution.png")))


## 6. Try it on your own headline

In [ ]:
from src.predict import predict

sample = "The central bank raised interest rates for the third time this year."
for item in predict(sample):
    print(f"{item['label']:10s} {item['score']:.3f}")


## 7. Summary & Insights

- Fine-tuned `bert-base-uncased` on AG News reaches ~94-95% accuracy and
  macro F1 on the full test set in typical reproductions of this benchmark.
- The most common confusion is **Business vs. Sci/Tech**, since articles
  about technology companies plausibly belong to either category — see the
  confusion matrix above.
- The confidence distribution shows whether the model is well-calibrated:
  correct predictions should cluster near 1.0 confidence, while
  misclassifications tend to have lower, more spread-out confidence — a
  useful signal for routing uncertain predictions to human review in a
  production system.
- `--quick` mode (3,000 rows, 1 epoch) validates the whole pipeline in
  minutes on CPU before committing to a full multi-hour fine-tune.

See `report.md` at the project root for the full write-up, and `README.md`
for setup and usage instructions. The fine-tuned model is deployed via
`app/streamlit_app.py` — run `streamlit run app/streamlit_app.py` for a
live demo.